In [1]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "rh": {
        "url": "http://localhost:24000/mcp",
        "transport": "streamable_http",
    }
})

outils = await client.get_tools()     # envoie tools/list au serveur

for outil in outils:
    print(f"🔧 {outil.name} : {outil.description}")
    print(f"   arguments : {outil.args}\n")

🔧 get_employee_infos : Renvoie le poste, le salaire (en MAD) et l'ancienneté (en années) d'un employé à partir de son nom.
   arguments : {'nom': {'title': 'Nom', 'type': 'string'}}

🔧 calculer_prime : Calcule la prime annuelle d'un employé : 5 % du salaire par tranche de 5 ans d'ancienneté (max 20 %).
   arguments : {'salaire': {'title': 'Salaire', 'type': 'number'}, 'anciennete': {'title': 'Anciennete', 'type': 'integer'}}



In [2]:
get_employee = next(o for o in outils if o.name == "get_employee_infos")
print(await get_employee.ainvoke({"nom": "Sara"}))

[{'type': 'text', 'text': '{\n  "nom": "Sara",\n  "poste": "Data Scientist",\n  "salaire": 18000,\n  "anciennete": 4\n}', 'id': 'lc_378e385b-c0ed-45f0-b5ee-32693f566d46'}]


In [3]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

llm = ChatOllama(model="qwen3:4b", temperature=0, reasoning=True)

agent = create_agent(
    model=llm,
    tools=outils,                  # les outils viennent du serveur MCP
    system_prompt=(
        "Tu es un assistant RH. Utilise les outils disponibles pour répondre. "
        "Réponds en français, brièvement."
    ),
)

async def demander(question):
    print(f"\n👤 {question}")
    resultat = await agent.ainvoke({"messages": [HumanMessage(question)]})
    for message in resultat["messages"]:
        for appel in getattr(message, "tool_calls", None) or []:
            print(f"   🔧 {appel['name']}({appel['args']})   ← exécuté sur le serveur MCP")
    print(f"🤖 {resultat['messages'][-1].content}")

In [4]:
await demander("Quel est le poste de Yassine ?")
await demander("Quelle est la prime annuelle de Mohamed ?")
await demander("Qui est Karim ?")


👤 Quel est le poste de Yassine ?
   🔧 get_employee_infos({'nom': 'Yassine'})   ← exécuté sur le serveur MCP
🤖 Chef de projet

👤 Quelle est la prime annuelle de Mohamed ?
   🔧 get_employee_infos({'nom': 'Mohamed'})   ← exécuté sur le serveur MCP
   🔧 calculer_prime({'salaire': 23000, 'anciennete': 12})   ← exécuté sur le serveur MCP
🤖 La prime annuelle de Mohamed est de 2 300 MAD.

👤 Qui est Karim ?
   🔧 get_employee_infos({'nom': 'Karim'})   ← exécuté sur le serveur MCP
🤖 Aucun employé nommé Karim.


In [5]:
print("Agent RH (MCP) — tape 'quit' pour arrêter")
while True:
    question = input("Vous : ")
    if question.strip().lower() in ("quit", "exit", ""):
        break
    await demander(question)

Agent RH (MCP) — tape 'quit' pour arrêter

👤 qui est karim
   🔧 get_employee_infos({'nom': 'karim'})   ← exécuté sur le serveur MCP
🤖 Désolé, aucun employé nommé Karim n'est enregistré dans le système.
